<a href="https://colab.research.google.com/github/Lyse-Claudia/lab-4-llm-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Lyse Claudia Irera
**Student ID:** 62602028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [24]:
import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name
#MODEL = "openai/gpt-oss-120b"

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [25]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response # Return the full response object

# TODO: Call it once with a simple question and print the answer.
response = ask_llm("Is Ghana in Africa")
print(response.choices[0].message.content)
# TODO: Print response.usage as well — how many tokens did your call consume?
print(response.usage)

Yes, Ghana is indeed a country located in West Africa. It is situated on the Atlantic coast, bordered by Côte d'Ivoire (Ivory Coast) to the west, Burkina Faso to the north, Togo to the east, and the Gulf of Guinea to the south. Ghana is a sovereign nation with a rich cultural heritage, diverse geography, and a population of around 31 million people. Would you like to know more about Ghana?
CompletionUsage(completion_tokens=95, prompt_tokens=45, total_tokens=140, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.036215035, prompt_time=0.002128795, completion_time=0.243313408, total_time=0.245442203)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1. The system roles sets the general rules for the model while the user role specify the actual task, like what a user need.

2. A token is the the atomic unit of text a model can read.

API providers bill per token because computing costs increase as the number of tokens increase. A request containing a few words should cost less than one that contains a thousands words because they use different amount of resources.


### Part 1.2 — Temperature: the randomness dial

In [26]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.
# temperature = 0.0
print("Answers at temperature = 0.0: ")
print("1: ")
response11 = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=0.0)
print(response11.choices[0].message.content)
print("2: ")
response12 = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=0.0)
print(response12.choices[0].message.content)
print("3: ")
response13 = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=0.0)
print(response13.choices[0].message.content)
print("4: ")
response14 = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=0.0)
print(response14.choices[0].message.content)
print("5: ")
response15 = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=0.0)
print(response15.choices[0].message.content)

# temperature = 1.2
print("Answers at temperature = 1.2: ")
print("1: ")
response21 = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=1.2)
print(response21.choices[0].message.content)
print("2: ")
response22 = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=1.2)
print(response22.choices[0].message.content)
print("3: ")
response23 = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=1.2)
print(response23.choices[0].message.content)
print("4: ")
response24 = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=1.2)
print(response24.choices[0].message.content)
print("5: ")
response22 = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=1.2)
print(response22.choices[0].message.content)





Answers at temperature = 0.0: 
1: 
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect". This name could appeal to market traders who want to collect and save their earnings.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could be attractive to market traders who need to access their savings quickly.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, which could convey a sense of community and cooperation among market traders.
6. **Kae Dua**: "Kae Dua" means "good savings" or "profitable savings" in the Akan language, which could appeal to market traders who want to grow their s

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**

AT temperature = 0.0, the answers were more or less the same, there is no big difference between the answers. At t = 1.2, the model started giving diverse answers to some extent. It means that the temperature controls how diverse or creative the answer are for the model. For a loan decision support, it will be better to use a temperature that is low, somewhere between 0.0 and 0.3 will be appropriate.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [27]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [28]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
letter_text1 = LETTERS["L002"]
letter_text2 = LETTERS["L006"]
SUMMARY_PROMPT_V1 = "Summarize this loan application: "
v1_summary1_response = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_text1}")
print("--- V1 Summary for L002 ---")
print(v1_summary1_response.choices[0].message.content)
v1_summary2_response = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_text2}")
print("\n--- V1 Summary for L006 ---")
print(v1_summary2_response.choices[0].message.content)



# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)

#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SYSTEM_PROMPT_V2 = "You are an assistant to a microfinance loan officer. Summarize loan applications based on the following constraints: factual, neutral, no invented details, 3-4 sentences."
USER_PROMPT_TEMPLATE_V2 = "Summarize this loan application:\n\n{letter_text}"

v2_summary1_response = ask_llm(USER_PROMPT_TEMPLATE_V2.format(letter_text=letter_text1), system_prompt=SYSTEM_PROMPT_V2, temperature=0.0)
print("\n--- V2 Summary for L002 (temp=0.0) ---")
print(v2_summary1_response.choices[0].message.content)

v2_summary2_response = ask_llm(USER_PROMPT_TEMPLATE_V2.format(letter_text=letter_text2), system_prompt=SYSTEM_PROMPT_V2, temperature=0.0)
print("\n--- V2 Summary for L006 (temp=0.0) ---")
print(v2_summary2_response.choices[0].message.content)

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("\n==Comparison: V1 vs V2 output==")
print("V1 has no fixed structure, L006 response came as bullets points "+
      "while L002 came as a prose.\n V2 is consistent and respect the length constraint")
print("V1 output use an informal language, with words such as pay off, claim, and contractions"+
      "\n while V2 use a more formal language like settle personal debts insted of pay off,"+
      " \nMr. Boateng, instead of the use of He in V1")


--- V1 Summary for L002 ---
Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. He needs the funds to repair his vehicle's engine and pay off personal debts. His business has been slow, but he expects it to improve after the festive season. He doesn't have collateral to offer and is relying on his future earnings to repay the loan. He's requesting urgent assistance.

--- V1 Summary for L006 ---
Here's a summary of the loan application:

* Applicant: Kofi, 22 years old
* Loan amount: GHS 50,000
* Proposed businesses: 
  1. Car washing business
  2. Provision shop
  3. Importing phones from Dubai
* Repayment plan: Pay back the loan in 1 year, once the businesses are successful
* Collateral: None, but Kofi claims to be trustworthy

Note: The application lacks concrete business plans, financial projections, and credit history, making it a high-risk loan.

--- V2 Summary for L002 (temp=0.0) ---
Kwame Boateng, a commercial driver in Kumasi, has applied for a l

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**

1. First problem solved V2 fixed is the inconsistent format and uncontrolled length(explained in the comparison found in the cell above)

2. The "no invented details" matter in this situation because the summary is going to be an important factor in critical decisions that involve money. The model cannot just add details that are not in the source letter. In LLM literature this failure mode is hallucination, which is when a model generates fluent and plausible looking content that is not from the source material.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [29]:
import json
import pandas as pd
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

EXTRACT_PROMPT = """You are an assistant that extracts structured data from microfinance loan application letters.

Return ONLY a JSON object with EXACTLY these keys: no extra keys, no missing keys, no explanatory text before or after the JSON:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

If a field is not stated in the letter, use null. Do not guess or infer values that are not explicitly written in the letter.
Example:
Letter:
"My name is Lehi Lena. I run a small tailoring shop in Berekuso and I am requesting a loan of GHS 8,000 to buy a new industrial sewing machine. My shop currently makes about GHS 600 profit a month. My brother has agreed to act as guarantor for this loan. I would like to repay it over 12 months."
Output:
{
  "applicant_name": "Ama Serwaa",
  "amount_ghs": 8000,
  "purpose": "buy a new industrial sewing machine",
  "monthly_profit_ghs": 600,
  "has_collateral_or_guarantor": true,
  "repayment_months": 12
}
"""

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

def extract_fields(letter_text, temperature = 0.0):
    raw_response_object = ask_llm(f"{EXTRACT_PROMPT}\n\nLetter:\n\n{letter_text}", temperature = temperature)
    raw_output = raw_response_object.choices[0].message.content.strip()
    response = raw_output
    if response.startswith("```json"):
      response = response[7:]
    elif response.startswith("```"):
      response = response[3:]

    if response.endswith("```"):
      response = response[:-3]
    response = response.strip()
    try:
        parsed = json.loads(response)
        return parsed
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")
        print(f"Raw model output was:\n{raw_output}")
        return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

rows = []
for letter_id, letter_text in LETTERS.items():
    parsed = extract_fields(letter_text)
    if parsed is None:
        #when extraction has failed
        row = {"letter_id": letter_id, "parse_failed": True}
    else:
        row = {"letter_id": letter_id, "parse_failed": False, **parsed}
    rows.append(row)

df = pd.DataFrame(rows)
df

,letter_id,parse_failed,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,False,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,False,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,False,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,False,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,False,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,False,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
1. The example should not come from the six letters because it would leak the answer. If the model already saw the correct output for one of the letters as an example, we can't know if it really understood how to extract from a new letter or if it just copied the pattern from the one we showed it. It is like giving the model the answer key before the test which might make the evaluation of the other letters biased.

2. Without that instruction, the model might try to guess or infer a value that sounds reasonable but is not actually written in the letter. For example, if a letter does not mention monthly profit, the model could try to fill in a plausible number instead of saying null, because it is trying to complete the JSON fully. In a loan application this is dangerous because a made-up number can look just as convincing as a real one.

3. The model become more creative and less predictable as the temperature increases. But this specific tasks does not require any creativity or inferences, it has to process input as given. So, choosing a temperature of 0 makes sure that the model provide more consinstent results

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [30]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.


BRIEF_PROMPT = """You are an assistant to a microfinance loan officer. Your job is to prepare a
briefing note that helps the officer evaluate a loan application. You do not make lending
decisions, you support instead. The final decisions are always made by a human loan officer. Your role is only to
organize information to support their judgment.

Based on the letter and the extracted data below, produce a recommendation brief for the loan officer that must have exactly these
four sections:

1. Strengths:
        bullet points, grounded only in the letter. Do not guess or infer any details.
2. Risks / Red Flags:
        bullet points highlighting anything that could concern a loan officer
        (e.g. no collateral, no experience, vague repayment plan, unrealistic assumptions).
3. Missing Information:
      bullet points listing what the officer should ask the applicant for,
      based on gaps in the letter or extracted data (e.g. fields that were null).
4. Suggested Next Step:
     ONE short recommendation for a process step, such as "invite for
     interview", "request supporting documents", or "flag for senior review". Do NOT recommend
     "approve" or "reject", that decision belongs to the human officer.

Letter:
{letter_text}

Extracted data (JSON):
{extracted_json}

Now write the briefing note using the four sections above.
"""

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
for letter_id, letter_text in LETTERS.items():
    row = df[df["letter_id"] == letter_id].iloc[0]

    if row["parse_failed"]:
        print(f"Skipping brief for {letter_id} — extraction failed earlier")
        continue

    extracted_data = row.drop(labels=["letter_id", "parse_failed"]).to_dict()
    extracted_json = json.dumps(extracted_data)

    brief_response = ask_llm(
        BRIEF_PROMPT.format(letter_text=letter_text, extracted_json=extracted_json),
        temperature=0.0
    )

    if letter_id in ["L001", "L002","L003", "L006"]:
        print(f"\n--- Brief for letter {letter_id} ---")
        print(brief_response.choices[0].message.content)


--- Brief for letter L001 ---
**Briefing Note for Loan Application**

### Strengths:
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market.
* She has a stable monthly profit of GHS 900 from her current stall.
* She has saved GHS 2,500 with the susu scheme over two years without missing a contribution.
* She has a guarantor, her sister, who is a teacher.

### Risks / Red Flags:
* The applicant's repayment plan of GHS 450 monthly over 20 months may be tight, considering her monthly profit is GHS 900, which might not leave much room for unexpected expenses or fluctuations in income.
* There is no detailed information on how the deep freezer and expansion into frozen foods will increase her profits to support the loan repayment.

### Missing Information:
* Detailed business plan or financial projections for the expanded business.
* Information on the sister's (guarantor's) financial stability and ability to take on the guarantee.
* Valuation or det

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**

1. The model designed the briefs the same way and correctly identified the strengths and red flags for each. It is proved by the way it formulated the next steps for each. For L003, the next steps are only about inviting the person for the interview to discuss the loan and clarify any concerns while for L006, it insist on inviting the person for the interview so that they can get more information on their own understanding of what they want to do, the risks and challenges involved. It shows that the model was able to differentiate a strong application from a week one.

2. The practical reason is that the model does not have all the information that banks consider before making a final decision. They might have their own business rules and most importantly a way to verify that what the applicant wrote is true. So it should not make a final call with incomplete information.
The ethical reason is that if a loan is approved or rejected, someone needs to be responsible for that decision, especially if something goes wrong later. A human can be held accountable, but the model cannot, so the decision must stay with a person. Also, the model could carry bias from its training data (e.g. treating some names, regions, or writing styles less favorably). Keeping the decision with a human means someone can notice and question unfair patterns instead of the bias being hidden inside an automatic approve/reject.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 2db23547fb6a90cd6f3ce5e9c5fbc73b9e80abd6


---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [31]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
          "has_collateral_or_guarantor", "repayment_months"]

gold_letter_ids = ["L001", "L003", "L006"]

def values_match(field, extracted_val, gold_val):
    if field == "applicant_name":
        # case-insensitive string match, guard against non-string/NaN
        if not isinstance(extracted_val, str):
            return gold_val is None
        return str(extracted_val).strip().lower() == str(gold_val).strip().lower()
    else:
        # exact match for numbers/booleans/null
        if isinstance(extracted_val, float) and pd.isna(extracted_val):
            extracted_val = None
        return extracted_val == gold_val

comparison_rows = []
for field in fields:
    row = {"field": field}
    match_count = 0
    for letter_id in gold_letter_ids:
        extracted_val = df.loc[df["letter_id"] == letter_id, field].values[0]
        gold_val = GOLD[letter_id][field]
        match = values_match(field, extracted_val, gold_val)
        if match:
          row[letter_id] = "yes"
        else:
          row[letter_id] = f"No ({extracted_val} vs {gold_val})"
        match_count += int(match)
    row["accuracy"] = f"{match_count}/{len(gold_letter_ids)}"
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df



,field,L001,L003,L006,accuracy
0,applicant_name,yes,yes,yes,3/3
1,amount_ghs,yes,yes,yes,3/3
2,purpose,No (buy a deep freezer and expand into frozen ...,No (purchase two industrial sewing machines an...,"No (start a car washing business, a provision ...",0/3
3,monthly_profit_ghs,yes,yes,yes,3/3
4,has_collateral_or_guarantor,yes,yes,yes,3/3
5,repayment_months,yes,yes,yes,3/3


### Part 4.2 — Reliability: is the system consistent?

In [32]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

# at temperature=0
results_t0 = []
valid_count_t0 = 0
unique_json_t0 = set()
print("=== Temperature = 0 ===")
for i in range(5):
    result = extract_fields(LETTERS["L004"], temperature=0)
    print(f"run {i+1}:")
    print(result)
    results_t0.append(result)
    if result is not None:
        valid_count_t0 += 1
        unique_json_t0.add(json.dumps(result, sort_keys=True))
print(f"Valid JSON: {valid_count_t0}/5")
print(f"Unique outputs among valid runs: {len(unique_json_t0)}")

# at temperature=1.0
print("\n=== Temperature = 1.0 ")
results_t1 = []
valid_count_t1 = 0
unique_json_t1 = set()
for i in range(5):
    result = extract_fields(LETTERS["L004"], temperature=1.0)
    print(f"run {i+1}:")
    print(result)
    results_t1.append(result)
    if result is not None:
        valid_count_t1 += 1
        unique_json_t1.add(json.dumps(result, sort_keys=True))

print(f"Valid JSON: {valid_count_t1}/5")
print(f"Unique outputs among valid runs: {len(unique_json_t1)}")



=== Temperature = 0 ===
run 1:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
run 2:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
run 3:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
run 4:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
run 5:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Valid JSON: 5/5
Uniqu

### Part 4.3 — Hallucination probing

In [33]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
test1 = ask_llm(f"Give me only the credit score for this letter: {LETTERS["L002"]}?", system_prompt=SYSTEM_PROMPT_V2, temperature=0.0)
print(test1.choices[0].message.content)
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?
text ="""Weather Report — Kumasi, Ashanti Region

Today's forecast: partly cloudy with a high of 29°C and a low of 21°C. Humidity is
expected around 78%, with a light chance of afternoon showers, particularly after 3pm.
Winds will be light, coming from the southwest at about 10 km/h. Visibility remains
good throughout the day. Tomorrow is expected to be similar, though slightly warmer,
with highs near 31°C. The rainy season continues into next month, so residents are
advised to carry umbrellas in the afternoons."""
test2 = extract_fields(text)

print()
print(test2)
# TODO: Record the outputs verbatim below and label each PASS or FAIL.

There is no credit score mentioned in the letter.

{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**

1. Five out of six fields reported an accuracy score of 3/3 while the purpose field failed completely(0/3). Thus, the purpose field was the hardest but given that this field is about paraphrasing, we can't say that it failed. There can be many valid ways to phrase the same idea.

2. At both temperature, extraction on L004 produced 5/5 valid JSON with only one unique output each run. The model was able to extract all the fields accurately.
There is no observable increase in variability or inconsistency when the temperature change.

3. The system did not hallucinate. It reported that the letter does not mention any credit score.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**

1.
 If the bank fully automated load decision by using this system there will be many different people who will be unfairly harmed. For example, people with little language and writing skills but who operate successful businesses, applicants from underserved communities, people who cannot afford professional help with applications, etc. The model may interpret poor english as a sign of a weak or risky application even when the applicant's business is really doing well.

2.

Using a third-party API comes with privacy, security, and data protection concerns. The data is basically stored and processed outside of the original country and people who have access to it might use it for their own benefits. Before deploying the system at a real Ghanaian institution, I would make sure that the API providers have clear laws and data protection protocols that comply with Ghana's data protection requirements. I would also check their security practices because if their centers get compromised, ours will be affected as well, which will affect our credibility in the Ghanian community.

3.
The first one will be a mandatory human review: the AI is just an assistand and it should only provide summaries and recommendations. A qualified loan officer will have to review the applications and consider all the other additional and contextual informations that the AI might not have considered or overlooked.

There also should be logging and monitoring. The system will log AI outputs, human decisions, errors, and the other informations so that the instutition can monitor for bias, inconsistent recommendations, and other defaults.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**
1.
Iterating on a prompt is similar to tuning the model's hyperparameters because they both are about changing settings, comparing results, and selecting what produce the best performance. The difference is that the hyperparameters affect the learning and/or the predicting process while prompt engineering is just changing the instructions and constraints to an already well trained model.

2.

I will not trust trust the system to run completely unattended because loan decisions are a very sensitive decisions that could also act on the credibility of the institution. In a world where topics of affirmative action, equity and equality, it is important that there is a human who is accountable for any decision made. The evaluation that influenced me the most is the language barrier, as someone who has experienced not being understood by people, it will be very unfair if my fate is going to be decided by an engineered system.

3.

The usage results show that one request used 45 prompt tokens and 104 completion tokens for a total of 149 tokens. For 1000 applications, this would be around 45000 prompt tokens + 104 000 completion tokens = 149000tokens. For the choice of a provider, the cost per input and output token will influence a lot to ensure that the institution can afford using the API, the other operational costs, and make profit. There should be other considerations too like the reliability, privacy, and security.

4.

Calling an API beat training my own model because the API has been trained well and has the ability we are looking for already. Training our own model will costs us more money, starting from getting large training datasets and acquiring computing resources, which will lead to long development process. It could shift the focus and investment into training the model instead of building the loan institution.

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.